#### Importing Libraries

In [1]:
# Data manipulation and numerical operations
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

# Sparse matrix operations
from scipy.sparse import load_npz

# Model development
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

# Model selection and hyperparameter tuning
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Experiment tracking
import mlflow
import mlflow.sklearn

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

print("All model development libraries imported successfully.")

All model development libraries imported successfully.


#### Loading Feature Engineering Artifacts

In [2]:
# Defining artifact directories
features_dir = Path("../data/processed/features")
models_dir = Path("../models")

# Loading word-level TF-IDF matrices
X_train_word = load_npz(features_dir / "X_train_word.npz")
X_test_word = load_npz(features_dir / "X_test_word.npz")

# Loading character-level TF-IDF matrices
X_train_char = load_npz(features_dir / "X_train_char.npz")
X_test_char = load_npz(features_dir / "X_test_char.npz")

# Loading combined TF-IDF matrices
X_train_combined = load_npz(features_dir / "X_train_combined.npz")
X_test_combined = load_npz(features_dir / "X_test_combined.npz")

# Loading encoded target labels
y_train = np.load(features_dir / "y_train.npy")
y_test = np.load(features_dir / "y_test.npy")

# Loading fitted transformers
word_vectorizer = joblib.load(models_dir / "word_tfidf_vectorizer.joblib")
char_vectorizer = joblib.load(models_dir / "char_tfidf_vectorizer.joblib")
label_encoder = joblib.load(models_dir / "label_encoder.joblib")

# Displaying loaded artifact dimensions
print("Feature Representation       Training Shape       Testing Shape")
print("-" * 68)

for name, X_train, X_test in [
    ("Word TF-IDF", X_train_word, X_test_word),
    ("Character TF-IDF", X_train_char, X_test_char),
    ("Combined TF-IDF", X_train_combined, X_test_combined)
]:
    print(f"{name:<28} {str(X_train.shape):<20} {X_test.shape}")

print(f"\nTraining target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")
print(f"Number of categories: {len(label_encoder.classes_)}")
print("\nAll feature engineering artifacts loaded successfully.")

Feature Representation       Training Shape       Testing Shape
--------------------------------------------------------------------
Word TF-IDF                  (28919, 30000)       (6391, 30000)
Character TF-IDF             (28919, 50000)       (6391, 50000)
Combined TF-IDF              (28919, 80000)       (6391, 80000)

Training target shape: (28919,)
Testing target shape: (6391,)
Number of categories: 10

All feature engineering artifacts loaded successfully.


**Observations on Loaded Feature Engineering Artifacts:**

- All three TF-IDF representations were loaded successfully, containing **30,000 word features**, **50,000 character features**, and **80,000 combined features**.
- The training and testing datasets contain **28,919** and **6,391** records, respectively, across **10 product categories**.
- The saved feature matrices, encoded targets, and fitted transformers are available for reproducible model development without repeating feature extraction.

#### Validating Training Data and Group Integrity

In [3]:
# Loading original training and testing records
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

# Extracting group identifiers
train_groups = train_df["Split_Group"].to_numpy()
test_groups = test_df["Split_Group"].to_numpy()

# Validating feature and target alignment
for X_train, X_test in [
    (X_train_word, X_test_word),
    (X_train_char, X_test_char),
    (X_train_combined, X_test_combined)
]:
    assert X_train.shape[0] == len(y_train) == len(train_groups)
    assert X_test.shape[0] == len(y_test) == len(test_groups)
    assert X_train.shape[1] == X_test.shape[1]

# Verifying encoded labels match the original records
assert np.array_equal(
    label_encoder.transform(train_df["Category_Label"]),
    y_train
)

assert np.array_equal(
    label_encoder.transform(test_df["Category_Label"]),
    y_test
)

# Verifying that groups do not overlap across partitions
train_group_set = set(train_groups)
test_group_set = set(test_groups)

assert train_group_set.isdisjoint(test_group_set)

# Displaying validation results
print(f"Training records: {len(train_df):,}")
print(f"Testing records: {len(test_df):,}")
print(f"Training groups: {len(train_group_set):,}")
print(f"Testing groups: {len(test_group_set):,}")
print(f"Overlapping groups: {len(train_group_set & test_group_set)}")
print(f"Number of target classes: {len(np.unique(y_train))}")

print("\nFeature alignment and group integrity verified successfully.")

Training records: 28,919
Testing records: 6,391
Training groups: 9,577
Testing groups: 2,395
Overlapping groups: 0
Number of target classes: 10

Feature alignment and group integrity verified successfully.


**Observations on Training Data and Group Integrity:**

- The training and testing partitions contain **28,919** and **6,391** records, respectively, with consistent feature and target alignment.
- The dataset contains **9,577 training groups** and **2,395 testing groups**, with **zero overlapping groups** between partitions.
- All **10 product categories** are represented in the training data, supporting multiclass model development and group-aware cross-validation.

#### Configuring MLflow Experiment Tracking

In [4]:
# Defining the MLflow tracking database
tracking_db = Path("../mlflow.db").resolve()

# Configuring MLflow tracking
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")

# Creating or selecting the project experiment
experiment_name = "Product Taxonomy Intelligence"

mlflow.set_experiment(experiment_name)

# Displaying experiment information
experiment = mlflow.get_experiment_by_name(experiment_name)

print(f"Experiment name: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Artifact location: {experiment.artifact_location}")

print("\nMLflow experiment tracking configured successfully.")

2026/10/28 20:59:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 20:59:27 INFO mlflow.store.db.utils: Updating database tables
2026/10/28 20:59:30 INFO mlflow.tracking.fluent: Experiment with name 'Product Taxonomy Intelligence' does not exist. Creating a new experiment.


Experiment name: Product Taxonomy Intelligence
Experiment ID: 1
Tracking URI: sqlite:///D:/Projects/ML and Data Science/Product Taxonomy Intelligence/mlflow.db
Artifact location: file:///d:/Projects/ML and Data Science/Product Taxonomy Intelligence/notebooks/mlruns/1

MLflow experiment tracking configured successfully.


#### Establishing the Majority-Class Baseline

In [5]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

# Configuring group-aware cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Defining baseline classifier
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Defining evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}

# Evaluating baseline using training data only
baseline_cv = cross_validate(
    estimator=baseline_model,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation results
baseline_results = {
    "Model": "Majority Class Baseline",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": baseline_cv["test_accuracy"].mean(),
    "Macro_F1": baseline_cv["test_f1_macro"].mean(),
    "Weighted_F1": baseline_cv["test_f1_weighted"].mean()
}

baseline_results_df = pd.DataFrame([baseline_results])

display(baseline_results_df.round(4))

print("\nCross-validation completed successfully.")

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Majority Class Baseline,Word TF-IDF,0.158,0.0273,0.0431



Cross-validation completed successfully.


**Observations on Majority-Class Baseline:**

- The majority-class baseline achieved **15.80% accuracy**, reflecting the limited performance of predicting only the most frequent product category.
- The **Macro F1-score of 0.0273** indicates extremely poor classification performance across the 10 product categories.
- The baseline establishes a minimum benchmark for evaluating whether subsequent machine learning models learn meaningful product classification patterns.

#### Logging Baseline Performance in MLflow

In [6]:
# Logging baseline experiment
with mlflow.start_run(run_name="Baseline_Majority_Class"):

    # Logging model configuration
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("feature_representation", "Word TF-IDF")
    mlflow.log_param("cv_strategy", "StratifiedGroupKFold")
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("random_state", RANDOM_STATE)

    # Logging cross-validation metrics
    mlflow.log_metric("cv_accuracy_mean", baseline_cv["test_accuracy"].mean())
    mlflow.log_metric("cv_macro_f1_mean", baseline_cv["test_f1_macro"].mean())
    mlflow.log_metric("cv_weighted_f1_mean", baseline_cv["test_f1_weighted"].mean())

    mlflow.log_metric("cv_accuracy_std", baseline_cv["test_accuracy"].std())
    mlflow.log_metric("cv_macro_f1_std", baseline_cv["test_f1_macro"].std())
    mlflow.log_metric("cv_weighted_f1_std", baseline_cv["test_f1_weighted"].std())

    # Logging baseline results as an artifact
    mlflow.log_table(
        baseline_results_df,
        artifact_file="baseline_results.json"
    )

print("Majority-class baseline logged successfully in MLflow.")

Majority-class baseline logged successfully in MLflow.


#### Training and Evaluating Logistic Regression

In [7]:
# Initializing Logistic Regression
lr_word = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
lr_word_cv = cross_validate(
    estimator=lr_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
lr_word_results = {
    "Model": "Logistic Regression",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": lr_word_cv["test_accuracy"].mean(),
    "Macro_F1": lr_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": lr_word_cv["test_f1_weighted"].mean()
}

lr_word_results_df = pd.DataFrame([lr_word_results])

display(lr_word_results_df.round(4))

print("\nLogistic Regression cross-validation completed successfully.")

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467



Logistic Regression cross-validation completed successfully.


**Observations on Logistic Regression (Word TF-IDF):**

- Logistic Regression achieved **94.65% accuracy** and a **Macro F1-score of 0.9488** using word-level TF-IDF features.
- The **Weighted F1-score of 0.9467** indicates strong overall classification performance across the product categories.
- Performance substantially exceeds the majority-class baseline (**Macro F1: 0.0273**), demonstrating that word-level features contain meaningful predictive information.

#### Logging Logistic Regression (Word TF-IDF) in MLflow

In [8]:
# Logging Logistic Regression with Word TF-IDF
with mlflow.start_run(run_name="LogisticRegression_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LogisticRegression",
        "feature_representation": "Word TF-IDF",
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = lr_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging experiment results
    mlflow.log_table(
        lr_word_results_df,
        artifact_file="logistic_regression_word_results.json"
    )

print("Logistic Regression (Word TF-IDF) logged successfully in MLflow.")

Logistic Regression (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Logistic Regression (Character TF-IDF)

In [9]:
# Initializing Logistic Regression
lr_char = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
lr_char_cv = cross_validate(
    estimator=lr_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
lr_char_results = {
    "Model": "Logistic Regression",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": lr_char_cv["test_accuracy"].mean(),
    "Macro_F1": lr_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": lr_char_cv["test_f1_weighted"].mean()
}

lr_char_results_df = pd.DataFrame([lr_char_results])

display(lr_char_results_df.round(4))

print("\nLogistic Regression (Character TF-IDF) cross-validation completed successfully.")

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762



Logistic Regression (Character TF-IDF) cross-validation completed successfully.


**Observations on Logistic Regression (Character TF-IDF):**

- Logistic Regression achieved **97.62% accuracy** and a **Macro F1-score of 0.9763** using character-level TF-IDF features.
- Character TF-IDF outperformed Word TF-IDF by **2.97 percentage points in accuracy** and **0.0275 in Macro F1-score**.
- The improvement suggests that character-level patterns, including product identifiers and naming variations, provide valuable information for product category classification.

#### Logging Logistic Regression (Character TF-IDF) in MLflow

In [10]:
# Logging Logistic Regression with Character TF-IDF
with mlflow.start_run(run_name="LogisticRegression_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LogisticRegression",
        "feature_representation": "Character TF-IDF",
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = lr_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging experiment results
    mlflow.log_table(
        lr_char_results_df,
        artifact_file="logistic_regression_char_results.json"
    )

print("Logistic Regression (Character TF-IDF) logged successfully in MLflow.")

Logistic Regression (Character TF-IDF) logged successfully in MLflow.


#### Initializing the Model Comparison DataFrame

In [11]:
# Consolidating completed model experiments
model_results = [
    baseline_results,
    lr_word_results,
    lr_char_results
]

# Creating the model comparison DataFrame
model_results_df = pd.DataFrame(model_results)

# Sorting models by Macro F1-score
model_results_df = model_results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

display(model_results_df.round(4))

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
1,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
2,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431


#### Training and Evaluating Logistic Regression (Combined TF-IDF)

In [13]:
# Initializing Logistic Regression
lr_combined = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
lr_combined_cv = cross_validate(
    estimator=lr_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
lr_combined_results = {
    "Model": "Logistic Regression",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": lr_combined_cv["test_accuracy"].mean(),
    "Macro_F1": lr_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": lr_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
lr_combined_results_df = pd.DataFrame([lr_combined_results])

print("Logistic Regression (Combined TF-IDF) Results:")
display(lr_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(lr_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nLogistic Regression cross-validation completed successfully.")

Logistic Regression (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
1,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
2,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
3,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
4,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Logistic Regression cross-validation completed successfully.


**Observations on Logistic Regression (Combined TF-IDF):**

- Logistic Regression achieved **97.53% accuracy** and a **Macro F1-score of 0.9752** using the combined 80,000-feature representation.
- Combined TF-IDF substantially outperformed Word TF-IDF but performed slightly below Character TF-IDF (**0.9763 Macro F1**).
- The results suggest that character-level features capture most of the useful predictive information, while additional word-level features provide no measurable improvement under the current model configuration.

#### Logging Logistic Regression (Combined TF-IDF) in MLflow

In [14]:
# Logging Logistic Regression with Combined TF-IDF
with mlflow.start_run(run_name="LogisticRegression_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LogisticRegression",
        "feature_representation": "Combined TF-IDF",
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = lr_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        lr_combined_results_df,
        artifact_file="logistic_regression_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Logistic Regression (Combined TF-IDF) logged successfully in MLflow.")

Logistic Regression (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Linear SVM (Word TF-IDF)

In [15]:
# Initializing Linear SVM
svm_word = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
svm_word_cv = cross_validate(
    estimator=svm_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
svm_word_results = {
    "Model": "Linear SVM",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": svm_word_cv["test_accuracy"].mean(),
    "Macro_F1": svm_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": svm_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
svm_word_results_df = pd.DataFrame([svm_word_results])

print("Linear SVM (Word TF-IDF) Results:")
display(svm_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(svm_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nLinear SVM (Word TF-IDF) cross-validation completed successfully.")

Linear SVM (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Word TF-IDF,0.9525,0.955,0.9526



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
1,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
2,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
3,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
4,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
5,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Linear SVM (Word TF-IDF) cross-validation completed successfully.


**Observations on Linear SVM (Word TF-IDF):**

- Linear SVM achieved **95.25% accuracy** and a **Macro F1-score of 0.9550** using word-level TF-IDF features.
- The model outperformed Logistic Regression on the same representation, improving Macro F1 from **0.9488 to 0.9550**.
- Despite this improvement, performance remains below the current best model, Logistic Regression with Character TF-IDF (**0.9763 Macro F1**).

#### Logging Linear SVM (Word TF-IDF) in MLflow

In [16]:
# Logging Linear SVM with Word TF-IDF
with mlflow.start_run(run_name="LinearSVM_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LinearSVC",
        "feature_representation": "Word TF-IDF",
        "C": 1.0,
        "max_iter": 5000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = svm_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        svm_word_results_df,
        artifact_file="linear_svm_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Linear SVM (Word TF-IDF) logged successfully in MLflow.")

Linear SVM (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Linear SVM (Character TF-IDF)

In [17]:
# Initializing Linear SVM
svm_char = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
svm_char_cv = cross_validate(
    estimator=svm_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
svm_char_results = {
    "Model": "Linear SVM",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": svm_char_cv["test_accuracy"].mean(),
    "Macro_F1": svm_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": svm_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
svm_char_results_df = pd.DataFrame([svm_char_results])

print("Linear SVM (Character TF-IDF) Results:")
display(svm_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(svm_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nLinear SVM (Character TF-IDF) cross-validation completed successfully.")

Linear SVM (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.986,0.9857,0.986



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
2,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
5,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
6,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Linear SVM (Character TF-IDF) cross-validation completed successfully.


**Observations on Linear SVM (Character TF-IDF):**

- Linear SVM achieved **98.60% accuracy** and a **Macro F1-score of 0.9857** using character-level TF-IDF features.
- Character TF-IDF substantially outperformed Word TF-IDF, improving Macro F1 from **0.9550 to 0.9857**.
- The model established a new best cross-validation result, surpassing Logistic Regression with Character TF-IDF (**0.9763 Macro F1**).

#### Logging Linear SVM (Character TF-IDF) in MLflow

In [18]:
# Logging Linear SVM with Character TF-IDF
with mlflow.start_run(run_name="LinearSVM_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LinearSVC",
        "feature_representation": "Character TF-IDF",
        "C": 1.0,
        "max_iter": 5000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = svm_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        svm_char_results_df,
        artifact_file="linear_svm_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Linear SVM (Character TF-IDF) logged successfully in MLflow.")

Linear SVM (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Linear SVM (Combined TF-IDF)

In [19]:
# Initializing Linear SVM
svm_combined = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
svm_combined_cv = cross_validate(
    estimator=svm_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
svm_combined_results = {
    "Model": "Linear SVM",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": svm_combined_cv["test_accuracy"].mean(),
    "Macro_F1": svm_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": svm_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
svm_combined_results_df = pd.DataFrame([svm_combined_results])

print("Linear SVM (Combined TF-IDF) Results:")
display(svm_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(svm_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nLinear SVM (Combined TF-IDF) cross-validation completed successfully.")

Linear SVM (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Combined TF-IDF,0.984,0.9837,0.984



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Linear SVM (Combined TF-IDF) cross-validation completed successfully.


**Observations on Linear SVM (Combined TF-IDF):**

- Linear SVM achieved **98.40% accuracy** and a **Macro F1-score of 0.9837** using the combined 80,000-feature representation.
- Combined TF-IDF substantially outperformed Word TF-IDF but remained slightly below Character TF-IDF (**0.9857 Macro F1**).
- Character-level features continue to provide the strongest results, suggesting that additional word-level features offer limited benefit under the current Linear SVM configuration.

#### Logging Linear SVM (Combined TF-IDF) in MLflow

In [20]:
# Logging Linear SVM with Combined TF-IDF
with mlflow.start_run(run_name="LinearSVM_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "LinearSVC",
        "feature_representation": "Combined TF-IDF",
        "C": 1.0,
        "max_iter": 5000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = svm_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        svm_combined_results_df,
        artifact_file="linear_svm_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Linear SVM (Combined TF-IDF) logged successfully in MLflow.")

Linear SVM (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Multinomial Naive Bayes (Word TF-IDF)

In [21]:
# Initializing Multinomial Naive Bayes
mnb_word = MultinomialNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
mnb_word_cv = cross_validate(
    estimator=mnb_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mnb_word_results = {
    "Model": "Multinomial Naive Bayes",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": mnb_word_cv["test_accuracy"].mean(),
    "Macro_F1": mnb_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": mnb_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mnb_word_results_df = pd.DataFrame([mnb_word_results])

print("Multinomial Naive Bayes (Word TF-IDF) Results:")
display(mnb_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mnb_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMultinomial Naive Bayes (Word TF-IDF) cross-validation completed successfully.")

Multinomial Naive Bayes (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Multinomial Naive Bayes,Word TF-IDF,0.9219,0.9198,0.9215



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Multinomial Naive Bayes,Word TF-IDF,0.9219,0.9198,0.9215
8,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Multinomial Naive Bayes (Word TF-IDF) cross-validation completed successfully.


#### Logging Multinomial Naive Bayes (Word TF-IDF) in MLflow

In [22]:
# Logging Multinomial Naive Bayes with Word TF-IDF
with mlflow.start_run(run_name="MultinomialNB_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MultinomialNB",
        "feature_representation": "Word TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mnb_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mnb_word_results_df,
        artifact_file="multinomial_nb_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Multinomial Naive Bayes (Word TF-IDF) logged successfully in MLflow.")

Multinomial Naive Bayes (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Multinomial Naive Bayes (Character TF-IDF)

In [23]:
# Initializing Multinomial Naive Bayes
mnb_char = MultinomialNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
mnb_char_cv = cross_validate(
    estimator=mnb_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mnb_char_results = {
    "Model": "Multinomial Naive Bayes",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": mnb_char_cv["test_accuracy"].mean(),
    "Macro_F1": mnb_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": mnb_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mnb_char_results_df = pd.DataFrame([mnb_char_results])

print("Multinomial Naive Bayes (Character TF-IDF) Results:")
display(mnb_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mnb_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMultinomial Naive Bayes (Character TF-IDF) cross-validation completed successfully.")

Multinomial Naive Bayes (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Multinomial Naive Bayes,Character TF-IDF,0.9422,0.9332,0.9393



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Multinomial Naive Bayes,Character TF-IDF,0.9422,0.9332,0.9393
8,Multinomial Naive Bayes,Word TF-IDF,0.9219,0.9198,0.9215
9,Majority Class Baseline,Word TF-IDF,0.1580,0.0273,0.0431



Multinomial Naive Bayes (Character TF-IDF) cross-validation completed successfully.


**Observations on Multinomial Naive Bayes (Character TF-IDF):**

- Multinomial Naive Bayes achieved **94.22% accuracy** and a **Macro F1-score of 0.9332** using character-level TF-IDF features.
- Character TF-IDF outperformed Word TF-IDF, improving Macro F1 from **0.9198 to 0.9332**.
- Despite the improvement, the model remains below the current best-performing Linear SVM with Character TF-IDF (**0.9857 Macro F1**).

#### Logging Multinomial Naive Bayes (Character TF-IDF) in MLflow

In [24]:
# Logging Multinomial Naive Bayes with Character TF-IDF
with mlflow.start_run(run_name="MultinomialNB_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MultinomialNB",
        "feature_representation": "Character TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mnb_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mnb_char_results_df,
        artifact_file="multinomial_nb_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Multinomial Naive Bayes (Character TF-IDF) logged successfully in MLflow.")

Multinomial Naive Bayes (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Multinomial Naive Bayes (Combined TF-IDF)

In [25]:
# Initializing Multinomial Naive Bayes
mnb_combined = MultinomialNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
mnb_combined_cv = cross_validate(
    estimator=mnb_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mnb_combined_results = {
    "Model": "Multinomial Naive Bayes",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": mnb_combined_cv["test_accuracy"].mean(),
    "Macro_F1": mnb_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": mnb_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mnb_combined_results_df = pd.DataFrame([mnb_combined_results])

print("Multinomial Naive Bayes (Combined TF-IDF) Results:")
display(mnb_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mnb_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMultinomial Naive Bayes (Combined TF-IDF) cross-validation completed successfully.")

Multinomial Naive Bayes (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.947



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.9470
8,Multinomial Naive Bayes,Character TF-IDF,0.9422,0.9332,0.9393
9,Multinomial Naive Bayes,Word TF-IDF,0.9219,0.9198,0.9215



Multinomial Naive Bayes (Combined TF-IDF) cross-validation completed successfully.


**Observations on Multinomial Naive Bayes (Combined TF-IDF):**

- Multinomial Naive Bayes achieved **94.87% accuracy** and a **Macro F1-score of 0.9433** using the combined TF-IDF representation.
- Combined TF-IDF outperformed both Word TF-IDF (**0.9198 Macro F1**) and Character TF-IDF (**0.9332 Macro F1**).
- Unlike Logistic Regression and Linear SVM, Multinomial Naive Bayes benefited from combining word-level and character-level features, although its performance remained below the leading linear classifiers.

#### Logging Multinomial Naive Bayes (Combined TF-IDF) in MLflow

In [26]:
# Logging Multinomial Naive Bayes with Combined TF-IDF
with mlflow.start_run(run_name="MultinomialNB_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MultinomialNB",
        "feature_representation": "Combined TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mnb_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mnb_combined_results_df,
        artifact_file="multinomial_nb_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Multinomial Naive Bayes (Combined TF-IDF) logged successfully in MLflow.")

Multinomial Naive Bayes (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Complement Naive Bayes (Word TF-IDF)

In [27]:
# Initializing Complement Naive Bayes
cnb_word = ComplementNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
cnb_word_cv = cross_validate(
    estimator=cnb_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
cnb_word_results = {
    "Model": "Complement Naive Bayes",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": cnb_word_cv["test_accuracy"].mean(),
    "Macro_F1": cnb_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": cnb_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
cnb_word_results_df = pd.DataFrame([cnb_word_results])

print("Complement Naive Bayes (Word TF-IDF) Results:")
display(cnb_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(cnb_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nComplement Naive Bayes (Word TF-IDF) cross-validation completed successfully.")

Complement Naive Bayes (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Complement Naive Bayes,Word TF-IDF,0.938,0.935,0.9375



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.9470
8,Complement Naive Bayes,Word TF-IDF,0.9380,0.9350,0.9375
9,Multinomial Naive Bayes,Character TF-IDF,0.9422,0.9332,0.9393



Complement Naive Bayes (Word TF-IDF) cross-validation completed successfully.


**Observations on Complement Naive Bayes (Word TF-IDF):**

- Complement Naive Bayes achieved **93.80% accuracy** and a **Macro F1-score of 0.9350** using word-level TF-IDF features.
- The model outperformed Multinomial Naive Bayes on the same representation, improving Macro F1 from **0.9198 to 0.9350**.
- Despite the improvement, performance remains below Logistic Regression (**0.9488 Macro F1**) and Linear SVM (**0.9550 Macro F1**) using Word TF-IDF.

#### Logging Complement Naive Bayes (Word TF-IDF) in MLflow

In [28]:
# Logging Complement Naive Bayes with Word TF-IDF
with mlflow.start_run(run_name="ComplementNB_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ComplementNB",
        "feature_representation": "Word TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = cnb_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        cnb_word_results_df,
        artifact_file="complement_nb_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Complement Naive Bayes (Word TF-IDF) logged successfully in MLflow.")

Complement Naive Bayes (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Complement Naive Bayes (Character TF-IDF)

In [29]:
# Initializing Complement Naive Bayes
cnb_char = ComplementNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
cnb_char_cv = cross_validate(
    estimator=cnb_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
cnb_char_results = {
    "Model": "Complement Naive Bayes",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": cnb_char_cv["test_accuracy"].mean(),
    "Macro_F1": cnb_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": cnb_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
cnb_char_results_df = pd.DataFrame([cnb_char_results])

print("Complement Naive Bayes (Character TF-IDF) Results:")
display(cnb_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(cnb_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nComplement Naive Bayes (Character TF-IDF) cross-validation completed successfully.")

Complement Naive Bayes (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Complement Naive Bayes,Character TF-IDF,0.9517,0.9429,0.9493



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
6,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
7,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.9470
8,Complement Naive Bayes,Character TF-IDF,0.9517,0.9429,0.9493
9,Complement Naive Bayes,Word TF-IDF,0.9380,0.9350,0.9375



Complement Naive Bayes (Character TF-IDF) cross-validation completed successfully.


**Observations on Complement Naive Bayes (Character TF-IDF):**

- Complement Naive Bayes achieved **95.17% accuracy** and a **Macro F1-score of 0.9429** using character-level TF-IDF features.
- Character TF-IDF outperformed Word TF-IDF, improving Macro F1 from **0.9350 to 0.9429**.
- The model also outperformed Multinomial Naive Bayes with Character TF-IDF (**0.9332 Macro F1**), although it remained below the leading Linear SVM model.

#### Logging Complement Naive Bayes (Character TF-IDF) in MLflow

In [30]:
# Logging Complement Naive Bayes with Character TF-IDF
with mlflow.start_run(run_name="ComplementNB_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ComplementNB",
        "feature_representation": "Character TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = cnb_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        cnb_char_results_df,
        artifact_file="complement_nb_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Complement Naive Bayes (Character TF-IDF) logged successfully in MLflow.")

Complement Naive Bayes (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Complement Naive Bayes (Combined TF-IDF)

In [31]:
# Initializing Complement Naive Bayes
cnb_combined = ComplementNB(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
cnb_combined_cv = cross_validate(
    estimator=cnb_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
cnb_combined_results = {
    "Model": "Complement Naive Bayes",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": cnb_combined_cv["test_accuracy"].mean(),
    "Macro_F1": cnb_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": cnb_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
cnb_combined_results_df = pd.DataFrame([cnb_combined_results])

print("Complement Naive Bayes (Combined TF-IDF) Results:")
display(cnb_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(cnb_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nComplement Naive Bayes (Combined TF-IDF) cross-validation completed successfully.")

Complement Naive Bayes (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
6,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
7,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
8,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.9470
9,Complement Naive Bayes,Character TF-IDF,0.9517,0.9429,0.9493



Complement Naive Bayes (Combined TF-IDF) cross-validation completed successfully.


**Observations on Complement Naive Bayes (Combined TF-IDF):**

- Complement Naive Bayes achieved **96.24% accuracy** and a **Macro F1-score of 0.9582** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF outperformed Word TF-IDF (**0.9350 Macro F1**) and Character TF-IDF (**0.9429 Macro F1**), making it the strongest representation for Complement Naive Bayes.
- The model also surpassed Multinomial Naive Bayes with Combined TF-IDF (**0.9433 Macro F1**), although Linear SVM with Character TF-IDF remains the overall leader.

#### Logging Complement Naive Bayes (Combined TF-IDF) in MLflow

In [32]:
# Logging Complement Naive Bayes with Combined TF-IDF
with mlflow.start_run(run_name="ComplementNB_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ComplementNB",
        "feature_representation": "Combined TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = cnb_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        cnb_combined_results_df,
        artifact_file="complement_nb_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Complement Naive Bayes (Combined TF-IDF) logged successfully in MLflow.")

Complement Naive Bayes (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Ridge Classifier (Word TF-IDF)

In [33]:
# Initializing Ridge Classifier
ridge_word = RidgeClassifier(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
ridge_word_cv = cross_validate(
    estimator=ridge_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
ridge_word_results = {
    "Model": "Ridge Classifier",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": ridge_word_cv["test_accuracy"].mean(),
    "Macro_F1": ridge_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": ridge_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
ridge_word_results_df = pd.DataFrame([ridge_word_results])

print("Ridge Classifier (Word TF-IDF) Results:")
display(ridge_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(ridge_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRidge Classifier (Word TF-IDF) cross-validation completed successfully.")

Ridge Classifier (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Word TF-IDF,0.953,0.9558,0.9532



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
1,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
2,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
3,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
6,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
7,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
8,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467
9,Multinomial Naive Bayes,Combined TF-IDF,0.9487,0.9433,0.9470



Ridge Classifier (Word TF-IDF) cross-validation completed successfully.


**Observations on Ridge Classifier (Word TF-IDF):**

- Ridge Classifier achieved **95.30% accuracy** and a **Macro F1-score of 0.9558** using word-level TF-IDF features.
- The model slightly outperformed Linear SVM (**0.9550 Macro F1**) and Logistic Regression (**0.9488 Macro F1**) on the same representation.
- Ridge Classifier is currently the strongest Word TF-IDF linear classifier, although Linear SVM with Character TF-IDF remains the overall leader.

#### Logging Ridge Classifier (Word TF-IDF) in MLflow

In [34]:
# Logging Ridge Classifier with Word TF-IDF
with mlflow.start_run(run_name="RidgeClassifier_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RidgeClassifier",
        "feature_representation": "Word TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = ridge_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        ridge_word_results_df,
        artifact_file="ridge_classifier_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Ridge Classifier (Word TF-IDF) logged successfully in MLflow.")

Ridge Classifier (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Ridge Classifier (Character TF-IDF)

In [35]:
# Initializing Ridge Classifier
ridge_char = RidgeClassifier(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
ridge_char_cv = cross_validate(
    estimator=ridge_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
ridge_char_results = {
    "Model": "Ridge Classifier",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": ridge_char_cv["test_accuracy"].mean(),
    "Macro_F1": ridge_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": ridge_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
ridge_char_results_df = pd.DataFrame([ridge_char_results])

print("Ridge Classifier (Character TF-IDF) Results:")
display(ridge_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(ridge_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRidge Classifier (Character TF-IDF) cross-validation completed successfully.")

Ridge Classifier (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
2,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
3,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
4,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
7,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
8,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526
9,Logistic Regression,Word TF-IDF,0.9465,0.9488,0.9467



Ridge Classifier (Character TF-IDF) cross-validation completed successfully.


**Observations on Ridge Classifier (Character TF-IDF):**

- Ridge Classifier achieved **98.78% accuracy** and a **Macro F1-score of 0.9877** using character-level TF-IDF features.
- The model outperformed Linear SVM (**0.9857 Macro F1**) and Logistic Regression (**0.9763 Macro F1**) using the same representation.
- Ridge Classifier with Character TF-IDF is now the **best-performing model overall**, demonstrating the effectiveness of character-level features for product category classification.

#### Logging Ridge Classifier (Character TF-IDF) in MLflow

In [36]:
# Logging Ridge Classifier with Character TF-IDF
with mlflow.start_run(run_name="RidgeClassifier_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RidgeClassifier",
        "feature_representation": "Character TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = ridge_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        ridge_char_results_df,
        artifact_file="ridge_classifier_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Ridge Classifier (Character TF-IDF) logged successfully in MLflow.")

Ridge Classifier (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Ridge Classifier (Combined TF-IDF)

In [37]:
# Initializing Ridge Classifier
ridge_combined = RidgeClassifier(
    alpha=1.0
)

# Evaluating using group-aware cross-validation
ridge_combined_cv = cross_validate(
    estimator=ridge_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
ridge_combined_results = {
    "Model": "Ridge Classifier",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": ridge_combined_cv["test_accuracy"].mean(),
    "Macro_F1": ridge_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": ridge_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
ridge_combined_results_df = pd.DataFrame([ridge_combined_results])

print("Ridge Classifier (Combined TF-IDF) Results:")
display(ridge_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(ridge_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRidge Classifier (Combined TF-IDF) cross-validation completed successfully.")

Ridge Classifier (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Ridge Classifier (Combined TF-IDF) cross-validation completed successfully.


**Observations on Ridge Classifier (Combined TF-IDF):**

- Ridge Classifier achieved **98.66% accuracy** and a **Macro F1-score of 0.9864** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF substantially outperformed Word TF-IDF (**0.9558 Macro F1**) but performed slightly below Character TF-IDF (**0.9877 Macro F1**).
- Character TF-IDF remains the strongest representation for Ridge Classifier and retains the highest overall cross-validation performance among the models evaluated.

#### Logging Ridge Classifier (Combined TF-IDF) in MLflow

In [38]:
# Logging Ridge Classifier with Combined TF-IDF
with mlflow.start_run(run_name="RidgeClassifier_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RidgeClassifier",
        "feature_representation": "Combined TF-IDF",
        "alpha": 1.0,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = ridge_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        ridge_combined_results_df,
        artifact_file="ridge_classifier_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Ridge Classifier (Combined TF-IDF) logged successfully in MLflow.")

Ridge Classifier (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Random Forest Classifier (Word TF-IDF)

In [39]:
# Initializing Random Forest Classifier
rf_word = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
rf_word_cv = cross_validate(
    estimator=rf_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
rf_word_results = {
    "Model": "Random Forest",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": rf_word_cv["test_accuracy"].mean(),
    "Macro_F1": rf_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": rf_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
rf_word_results_df = pd.DataFrame([rf_word_results])

print("Random Forest (Word TF-IDF) Results:")
display(rf_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(rf_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(by="Macro_F1", ascending=False)
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRandom Forest (Word TF-IDF) cross-validation completed successfully.")

Random Forest (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Random Forest,Word TF-IDF,0.8757,0.8706,0.8752



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Random Forest (Word TF-IDF) cross-validation completed successfully.


**Observations on Random Forest (Word TF-IDF):**

- Random Forest achieved **87.57% accuracy** and a **Macro F1-score of 0.8706** using word-level TF-IDF features.
- The model performed below Ridge Classifier (**0.9558 Macro F1**) and Linear SVM (**0.9550 Macro F1**) on the same representation.
- The results suggest that the current Random Forest configuration is less effective than linear classifiers for high-dimensional, sparse product-title features.

#### Logging Random Forest (Word TF-IDF) in MLflow

In [40]:
# Logging Random Forest with Word TF-IDF
with mlflow.start_run(run_name="RandomForest_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "feature_representation": "Word TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = rf_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        rf_word_results_df,
        artifact_file="random_forest_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Random Forest (Word TF-IDF) logged successfully in MLflow.")

Random Forest (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Random Forest (Character TF-IDF)

In [41]:
# Initializing Random Forest Classifier
rf_char = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
rf_char_cv = cross_validate(
    estimator=rf_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
rf_char_results = {
    "Model": "Random Forest",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": rf_char_cv["test_accuracy"].mean(),
    "Macro_F1": rf_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": rf_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
rf_char_results_df = pd.DataFrame([rf_char_results])

print("Random Forest (Character TF-IDF) Results:")
display(rf_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(rf_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRandom Forest (Character TF-IDF) cross-validation completed successfully.")

Random Forest (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Random Forest,Character TF-IDF,0.9353,0.9384,0.9372



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Random Forest (Character TF-IDF) cross-validation completed successfully.


**Observations on Random Forest (Character TF-IDF):**

- Random Forest achieved **93.53% accuracy** and a **Macro F1-score of 0.9384** using character-level TF-IDF features.
- Character TF-IDF substantially outperformed Word TF-IDF, improving Macro F1 from **0.8706 to 0.9384**.
- Despite the improvement, Random Forest remained below Ridge Classifier (**0.9877 Macro F1**) and Linear SVM (**0.9857 Macro F1**) using Character TF-IDF.

#### Logging Random Forest (Character TF-IDF) in MLflow

In [42]:
# Logging Random Forest with Character TF-IDF
with mlflow.start_run(run_name="RandomForest_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "feature_representation": "Character TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = rf_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        rf_char_results_df,
        artifact_file="random_forest_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Random Forest (Character TF-IDF) logged successfully in MLflow.")

Random Forest (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Random Forest (Combined TF-IDF)

In [43]:
# Initializing Random Forest Classifier
rf_combined = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
rf_combined_cv = cross_validate(
    estimator=rf_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
rf_combined_results = {
    "Model": "Random Forest",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": rf_combined_cv["test_accuracy"].mean(),
    "Macro_F1": rf_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": rf_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
rf_combined_results_df = pd.DataFrame([rf_combined_results])

print("Random Forest (Combined TF-IDF) Results:")
display(rf_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(rf_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nRandom Forest (Combined TF-IDF) cross-validation completed successfully.")

Random Forest (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Random Forest,Combined TF-IDF,0.9323,0.9364,0.9347



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Random Forest (Combined TF-IDF) cross-validation completed successfully.


**Observations on Random Forest (Combined TF-IDF):**

- Random Forest achieved **93.23% accuracy** and a **Macro F1-score of 0.9364** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF substantially outperformed Word TF-IDF (**0.8706 Macro F1**) but performed slightly below Character TF-IDF (**0.9384 Macro F1**).
- Character TF-IDF remains the strongest representation for Random Forest, although the model continues to trail the leading linear classifiers.

#### Logging Random Forest (Combined TF-IDF) in MLflow

In [44]:
# Logging Random Forest with Combined TF-IDF
with mlflow.start_run(run_name="RandomForest_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "feature_representation": "Combined TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = rf_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        rf_combined_results_df,
        artifact_file="random_forest_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Random Forest (Combined TF-IDF) logged successfully in MLflow.")

Random Forest (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Extra Trees Classifier (Word TF-IDF)

In [45]:
# Initializing Extra Trees Classifier
et_word = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
et_word_cv = cross_validate(
    estimator=et_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
et_word_results = {
    "Model": "Extra Trees",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": et_word_cv["test_accuracy"].mean(),
    "Macro_F1": et_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": et_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
et_word_results_df = pd.DataFrame([et_word_results])

print("Extra Trees (Word TF-IDF) Results:")
display(et_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(et_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nExtra Trees (Word TF-IDF) cross-validation completed successfully.")

Extra Trees (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Extra Trees,Word TF-IDF,0.8066,0.7975,0.81



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Extra Trees (Word TF-IDF) cross-validation completed successfully.


**Observations on Extra Trees (Word TF-IDF):**

- Extra Trees achieved **80.66% accuracy** and a **Macro F1-score of 0.7975** using word-level TF-IDF features.
- The model underperformed Random Forest (**0.8706 Macro F1**) using the same feature representation.
- The results indicate that the current Extra Trees configuration is less effective for sparse word-level features, with further evaluation needed to assess character-level representations.

#### Logging Extra Trees Classifier (Word TF-IDF) in MLflow

In [46]:
# Logging Extra Trees with Word TF-IDF
with mlflow.start_run(run_name="ExtraTrees_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "feature_representation": "Word TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = et_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        et_word_results_df,
        artifact_file="extra_trees_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Extra Trees (Word TF-IDF) logged successfully in MLflow.")

Extra Trees (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Extra Trees Classifier (Character TF-IDF)

In [47]:
# Initializing Extra Trees Classifier
et_char = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
et_char_cv = cross_validate(
    estimator=et_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
et_char_results = {
    "Model": "Extra Trees",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": et_char_cv["test_accuracy"].mean(),
    "Macro_F1": et_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": et_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
et_char_results_df = pd.DataFrame([et_char_results])

print("Extra Trees (Character TF-IDF) Results:")
display(et_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(et_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nExtra Trees (Character TF-IDF) cross-validation completed successfully.")

Extra Trees (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Extra Trees,Character TF-IDF,0.9326,0.9358,0.9347



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Extra Trees (Character TF-IDF) cross-validation completed successfully.


**Observations on Extra Trees (Character TF-IDF):**

- Extra Trees achieved **93.26% accuracy** and a **Macro F1-score of 0.9358** using character-level TF-IDF features.
- Character TF-IDF substantially outperformed Word TF-IDF, improving Macro F1 from **0.7975 to 0.9358**.
- The model approached Random Forest's performance (**0.9384 Macro F1**) using Character TF-IDF but remained below the leading linear classifiers.

#### Logging Extra Trees Classifier (Character TF-IDF) in MLflow

In [48]:
# Logging Extra Trees with Character TF-IDF
with mlflow.start_run(run_name="ExtraTrees_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "feature_representation": "Character TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = et_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        et_char_results_df,
        artifact_file="extra_trees_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Extra Trees (Character TF-IDF) logged successfully in MLflow.")

Extra Trees (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Extra Trees Classifier (Combined TF-IDF)

In [49]:
# Initializing Extra Trees Classifier
et_combined = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
et_combined_cv = cross_validate(
    estimator=et_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
et_combined_results = {
    "Model": "Extra Trees",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": et_combined_cv["test_accuracy"].mean(),
    "Macro_F1": et_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": et_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
et_combined_results_df = pd.DataFrame([et_combined_results])

print("Extra Trees (Combined TF-IDF) Results:")
display(et_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(et_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nExtra Trees (Combined TF-IDF) cross-validation completed successfully.")

Extra Trees (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Extra Trees,Combined TF-IDF,0.928,0.9303,0.9294



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



Extra Trees (Combined TF-IDF) cross-validation completed successfully.


**Observations on Extra Trees (Combined TF-IDF):**

- Extra Trees achieved **92.80% accuracy** and a **Macro F1-score of 0.9303** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF substantially outperformed Word TF-IDF (**0.7975 Macro F1**) but performed slightly below Character TF-IDF (**0.9358 Macro F1**).
- Character TF-IDF remains the strongest representation for Extra Trees, while the model continues to trail Random Forest and the leading linear classifiers.

#### Logging Extra Trees Classifier (Combined TF-IDF) in MLflow

In [51]:
# Logging Extra Trees with Combined TF-IDF
with mlflow.start_run(run_name="ExtraTrees_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "feature_representation": "Combined TF-IDF",
        "n_estimators": 100,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = et_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        et_combined_results_df,
        artifact_file="extra_trees_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Extra Trees (Combined TF-IDF) logged successfully in MLflow.")

Extra Trees (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating MLP Classifier (Word TF-IDF)

In [52]:
# Initializing MLP Classifier
mlp_word = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
mlp_word_cv = cross_validate(
    estimator=mlp_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mlp_word_results = {
    "Model": "MLP Classifier",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": mlp_word_cv["test_accuracy"].mean(),
    "Macro_F1": mlp_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": mlp_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mlp_word_results_df = pd.DataFrame([mlp_word_results])

print("MLP Classifier (Word TF-IDF) Results:")
display(mlp_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mlp_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMLP Classifier (Word TF-IDF) cross-validation completed successfully.")


MLP Classifier (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,MLP Classifier,Word TF-IDF,0.9491,0.951,0.9493



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
4,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
5,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
8,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532
9,Linear SVM,Word TF-IDF,0.9525,0.9550,0.9526



MLP Classifier (Word TF-IDF) cross-validation completed successfully.


**Observations on MLP Classifier (Word TF-IDF):**

- MLP Classifier achieved **94.91% accuracy** and a **Macro F1-score of 0.9510** using word-level TF-IDF features.
- The model slightly outperformed Logistic Regression (**0.9488 Macro F1**) but remained below Ridge Classifier (**0.9558**) and Linear SVM (**0.9550**).
- The neural network substantially outperformed Random Forest and Extra Trees, demonstrating competitive performance on sparse text features.

#### Logging MLP Classifier (Word TF-IDF) in MLflow

In [53]:
# Logging MLP Classifier with Word TF-IDF
with mlflow.start_run(run_name="MLPClassifier_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MLPClassifier",
        "feature_representation": "Word TF-IDF",
        "hidden_layer_sizes": "(64,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.001,
        "batch_size": 256,
        "learning_rate_init": 0.001,
        "max_iter": 30,
        "early_stopping": True,
        "n_iter_no_change": 5,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mlp_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mlp_word_results_df,
        artifact_file="mlp_classifier_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("MLP Classifier (Word TF-IDF) logged successfully in MLflow.")

MLP Classifier (Word TF-IDF) logged successfully in MLflow.


#### Training and Evaluating MLP Classifier (Character TF-IDF)

In [54]:
# Initializing MLP Classifier
mlp_char = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
mlp_char_cv = cross_validate(
    estimator=mlp_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mlp_char_results = {
    "Model": "MLP Classifier",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": mlp_char_cv["test_accuracy"].mean(),
    "Macro_F1": mlp_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": mlp_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mlp_char_results_df = pd.DataFrame([mlp_char_results])

print("MLP Classifier (Character TF-IDF) Results:")
display(mlp_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mlp_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMLP Classifier (Character TF-IDF) cross-validation completed successfully.")

MLP Classifier (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
4,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
5,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
6,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
7,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
8,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614
9,Ridge Classifier,Word TF-IDF,0.9530,0.9558,0.9532



MLP Classifier (Character TF-IDF) cross-validation completed successfully.


**Observations on MLP Classifier (Character TF-IDF):**

- MLP Classifier achieved **98.47% accuracy** and a **Macro F1-score of 0.9845** using character-level TF-IDF features.
- Character TF-IDF substantially outperformed Word TF-IDF, improving Macro F1 from **0.9510 to 0.9845**.
- The model demonstrated competitive performance against leading linear classifiers, although Ridge Classifier (**0.9877 Macro F1**) and Linear SVM (**0.9857 Macro F1**) remained ahead.

#### Logging MLP Classifier (Character TF-IDF) in MLflow

In [55]:
# Logging MLP Classifier with Character TF-IDF
with mlflow.start_run(run_name="MLPClassifier_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MLPClassifier",
        "feature_representation": "Character TF-IDF",
        "hidden_layer_sizes": "(64,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.001,
        "batch_size": 256,
        "learning_rate_init": 0.001,
        "max_iter": 30,
        "early_stopping": True,
        "n_iter_no_change": 5,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mlp_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mlp_char_results_df,
        artifact_file="mlp_classifier_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("MLP Classifier (Character TF-IDF) logged successfully in MLflow.")

MLP Classifier (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating MLP Classifier (Combined TF-IDF)

In [57]:
# Initializing MLP Classifier
mlp_combined = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=RANDOM_STATE
)

# Evaluating using group-aware cross-validation
mlp_combined_cv = cross_validate(
    estimator=mlp_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
mlp_combined_results = {
    "Model": "MLP Classifier",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": mlp_combined_cv["test_accuracy"].mean(),
    "Macro_F1": mlp_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": mlp_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
mlp_combined_results_df = pd.DataFrame([mlp_combined_results])

print("MLP Classifier (Combined TF-IDF) Results:")
display(mlp_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(mlp_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nMLP Classifier (Combined TF-IDF) cross-validation completed successfully.")

MLP Classifier (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
4,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
5,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
6,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
7,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
8,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
9,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614



MLP Classifier (Combined TF-IDF) cross-validation completed successfully.


**Observations on MLP Classifier (Combined TF-IDF):**

- MLP Classifier achieved **98.45% accuracy** and a **Macro F1-score of 0.9843** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF substantially outperformed Word TF-IDF (**0.9510 Macro F1**) but performed marginally below Character TF-IDF (**0.9845 Macro F1**).
- Character TF-IDF remains the strongest representation for MLP, suggesting that combining word-level features offers limited additional predictive benefit.

#### Logging MLP Classifier (Combined TF-IDF) in MLflow

In [58]:
# Logging MLP Classifier with Combined TF-IDF
with mlflow.start_run(run_name="MLPClassifier_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "MLPClassifier",
        "feature_representation": "Combined TF-IDF",
        "hidden_layer_sizes": "(64,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.001,
        "batch_size": 256,
        "learning_rate_init": 0.001,
        "max_iter": 30,
        "early_stopping": True,
        "n_iter_no_change": 5,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = mlp_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        mlp_combined_results_df,
        artifact_file="mlp_classifier_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("MLP Classifier (Combined TF-IDF) logged successfully in MLflow.")

MLP Classifier (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating SGD Classifier (Word TF-IDF)

In [59]:
from sklearn.linear_model import SGDClassifier

# Initializing SGD Classifier
sgd_word = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
sgd_word_cv = cross_validate(
    estimator=sgd_word,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
sgd_word_results = {
    "Model": "SGD Classifier",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": sgd_word_cv["test_accuracy"].mean(),
    "Macro_F1": sgd_word_cv["test_f1_macro"].mean(),
    "Weighted_F1": sgd_word_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
sgd_word_results_df = pd.DataFrame([sgd_word_results])

print("SGD Classifier (Word TF-IDF) Results:")
display(sgd_word_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(sgd_word_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nSGD Classifier (Word TF-IDF) cross-validation completed successfully.")

SGD Classifier (Word TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,SGD Classifier,Word TF-IDF,0.9506,0.9524,0.9509



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
4,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
5,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
6,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
7,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
8,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
9,Complement Naive Bayes,Combined TF-IDF,0.9624,0.9582,0.9614



SGD Classifier (Word TF-IDF) cross-validation completed successfully.


**Observations on SGD Classifier (Word TF-IDF):**

- SGD Classifier achieved **95.06% accuracy** and a **Macro F1-score of 0.9524** using word-level TF-IDF features.
- The model outperformed Logistic Regression (**0.9488 Macro F1**) but remained slightly below Ridge Classifier (**0.9558**) and Linear SVM (**0.9550**).
- SGD Classifier demonstrated competitive performance among linear models, supporting its suitability for high-dimensional, sparse text classification.

#### Logging SGD Classifier (Word TF-IDF) in MLflow

In [60]:
# Logging SGD Classifier with Word TF-IDF
with mlflow.start_run(run_name="SGDClassifier_Word_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "SGDClassifier",
        "feature_representation": "Word TF-IDF",
        "loss": "hinge",
        "penalty": "l2",
        "alpha": 0.0001,
        "max_iter": 1000,
        "tol": 1e-3,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = sgd_word_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        sgd_word_results_df,
        artifact_file="sgd_classifier_word_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("SGD Classifier (Word TF-IDF) logged successfully in MLflow.")

SGD Classifier (Word TF-IDF) logged successfully in MLflow.


In [61]:
#### Training and Evaluating SGD Classifier (Character TF-IDF)

In [62]:
# Initializing SGD Classifier
sgd_char = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
sgd_char_cv = cross_validate(
    estimator=sgd_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
sgd_char_results = {
    "Model": "SGD Classifier",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": sgd_char_cv["test_accuracy"].mean(),
    "Macro_F1": sgd_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": sgd_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
sgd_char_results_df = pd.DataFrame([sgd_char_results])

print("SGD Classifier (Character TF-IDF) Results:")
display(sgd_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(sgd_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nSGD Classifier (Character TF-IDF) cross-validation completed successfully.")

SGD Classifier (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,SGD Classifier,Character TF-IDF,0.9822,0.9822,0.9822



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
4,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
5,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
6,SGD Classifier,Character TF-IDF,0.9822,0.9822,0.9822
7,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
8,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753
9,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753



SGD Classifier (Character TF-IDF) cross-validation completed successfully.


**Observations on SGD Classifier (Character TF-IDF):**

- SGD Classifier achieved **98.22% accuracy** and a **Macro F1-score of 0.9822** using character-level TF-IDF features.
- Character TF-IDF substantially outperformed Word TF-IDF, improving Macro F1 from **0.9524 to 0.9822**.
- The model demonstrated strong classification performance but remained below Ridge Classifier (**0.9877 Macro F1**) and Linear SVM (**0.9857 Macro F1**) using the same representation.

#### Logging SGD Classifier (Character TF-IDF) in MLflow

In [63]:
# Logging SGD Classifier with Character TF-IDF
with mlflow.start_run(run_name="SGDClassifier_Character_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "SGDClassifier",
        "feature_representation": "Character TF-IDF",
        "loss": "hinge",
        "penalty": "l2",
        "alpha": 0.0001,
        "max_iter": 1000,
        "tol": 1e-3,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = sgd_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        sgd_char_results_df,
        artifact_file="sgd_classifier_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("SGD Classifier (Character TF-IDF) logged successfully in MLflow.")

SGD Classifier (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating SGD Classifier (Combined TF-IDF)

In [64]:
# Initializing SGD Classifier
sgd_combined = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Evaluating using group-aware cross-validation
sgd_combined_cv = cross_validate(
    estimator=sgd_combined,
    X=X_train_combined,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
sgd_combined_results = {
    "Model": "SGD Classifier",
    "Feature_Representation": "Combined TF-IDF",
    "Accuracy": sgd_combined_cv["test_accuracy"].mean(),
    "Macro_F1": sgd_combined_cv["test_f1_macro"].mean(),
    "Weighted_F1": sgd_combined_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
sgd_combined_results_df = pd.DataFrame([sgd_combined_results])

print("SGD Classifier (Combined TF-IDF) Results:")
display(sgd_combined_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(sgd_combined_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nSGD Classifier (Combined TF-IDF) cross-validation completed successfully.")

SGD Classifier (Combined TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,SGD Classifier,Combined TF-IDF,0.9823,0.9822,0.9823



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
3,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
4,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
5,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
6,SGD Classifier,Character TF-IDF,0.9822,0.9822,0.9822
7,SGD Classifier,Combined TF-IDF,0.9823,0.9822,0.9823
8,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762
9,Logistic Regression,Combined TF-IDF,0.9753,0.9752,0.9753



SGD Classifier (Combined TF-IDF) cross-validation completed successfully.


**Observations on SGD Classifier (Combined TF-IDF):**

- SGD Classifier achieved **98.23% accuracy** and a **Macro F1-score of 0.9822** using combined word-level and character-level TF-IDF features.
- Combined TF-IDF substantially outperformed Word TF-IDF (**0.9524 Macro F1**) and matched Character TF-IDF (**0.9822 Macro F1**).
- Combining word-level and character-level features provided negligible additional benefit, with Ridge Classifier (**0.9877 Macro F1**) remaining the overall leading model.

#### Logging SGD Classifier (Combined TF-IDF) in MLflow

In [65]:
# Logging SGD Classifier with Combined TF-IDF
with mlflow.start_run(run_name="SGDClassifier_Combined_TFIDF"):

    # Logging model configuration
    mlflow.log_params({
        "model": "SGDClassifier",
        "feature_representation": "Combined TF-IDF",
        "loss": "hinge",
        "penalty": "l2",
        "alpha": 0.0001,
        "max_iter": 1000,
        "tol": 1e-3,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = sgd_combined_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual model results
    mlflow.log_table(
        sgd_combined_results_df,
        artifact_file="sgd_classifier_combined_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("SGD Classifier (Combined TF-IDF) logged successfully in MLflow.")

SGD Classifier (Combined TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Voting Classifier (Character TF-IDF)

In [66]:
from sklearn.ensemble import VotingClassifier

# Initializing Voting Classifier
voting_char = VotingClassifier(
    estimators=[
        (
            "ridge",
            RidgeClassifier(
                alpha=1.0
            )
        ),
        (
            "linear_svm",
            LinearSVC(
                random_state=RANDOM_STATE,
                max_iter=5000
            )
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ],
    voting="hard",
    n_jobs=1
)

# Evaluating using group-aware cross-validation
voting_char_cv = cross_validate(
    estimator=voting_char,
    X=X_train_char,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation performance
voting_char_results = {
    "Model": "Voting Classifier",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": voting_char_cv["test_accuracy"].mean(),
    "Macro_F1": voting_char_cv["test_f1_macro"].mean(),
    "Weighted_F1": voting_char_cv["test_f1_weighted"].mean()
}

# Displaying individual model results
voting_char_results_df = pd.DataFrame([voting_char_results])

print("Voting Classifier (Character TF-IDF) Results:")
display(voting_char_results_df.round(4))

# Updating the centralized model comparison DataFrame
model_results.append(voting_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        by="Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

# Displaying updated model comparison
print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nVoting Classifier (Character TF-IDF) cross-validation completed successfully.")

Voting Classifier (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Voting Classifier,Character TF-IDF,0.9862,0.986,0.9862



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
2,Voting Classifier,Character TF-IDF,0.9862,0.9860,0.9862
3,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
4,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
5,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
6,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
7,SGD Classifier,Character TF-IDF,0.9822,0.9822,0.9822
8,SGD Classifier,Combined TF-IDF,0.9823,0.9822,0.9823
9,Logistic Regression,Character TF-IDF,0.9762,0.9763,0.9762



Voting Classifier (Character TF-IDF) cross-validation completed successfully.


**Observations on Voting Classifier (Character TF-IDF):**

- Voting Classifier achieved **98.62% accuracy** and a **Macro F1-score of 0.9860** using character-level TF-IDF features.
- The ensemble marginally outperformed Linear SVM (**0.9857 Macro F1**) and substantially outperformed Logistic Regression (**0.9763 Macro F1**).
- Hard voting did not surpass Ridge Classifier (**0.9877 Macro F1**), indicating limited additional benefit from combining these linear classifiers.

#### Logging Voting Classifier (Character TF-IDF) in MLflow

In [67]:
# Logging Voting Classifier with Character TF-IDF
with mlflow.start_run(run_name="VotingClassifier_Character_TFIDF"):

    # Logging ensemble configuration
    mlflow.log_params({
        "model": "VotingClassifier",
        "feature_representation": "Character TF-IDF",
        "voting": "hard",
        "base_estimators": "RidgeClassifier, LinearSVC, LogisticRegression",
        "ridge_alpha": 1.0,
        "linear_svm_max_iter": 5000,
        "logistic_regression_max_iter": 1000,
        "random_state": RANDOM_STATE,
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = voting_char_cv[f"test_{metric}"]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual ensemble results
    mlflow.log_table(
        voting_char_results_df,
        artifact_file="voting_classifier_char_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Voting Classifier (Character TF-IDF) logged successfully in MLflow.")

Voting Classifier (Character TF-IDF) logged successfully in MLflow.


#### Training and Evaluating Stacking Classifier (Character TF-IDF)

In [68]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import csr_matrix, hstack

# Defining base estimators
stacking_base_estimators = [
    ("ridge", RidgeClassifier(alpha=1.0)),
    ("linear_svm", LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    )),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
]

# Defining the meta-classifier
stacking_meta_estimator = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Configuring outer group-aware cross-validation
outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Configuring inner group-aware cross-validation
inner_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

stacking_fold_results = []

# Evaluating the stacking ensemble
for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(X_train_char, y_train, groups=train_groups),
    start=1
):
    X_outer_train = X_train_char[train_idx]
    X_outer_valid = X_train_char[valid_idx]

    y_outer_train = y_train[train_idx]
    y_outer_valid = y_train[valid_idx]

    groups_outer_train = train_groups[train_idx]

    n_classes = len(np.unique(y_train))
    n_models = len(stacking_base_estimators)

    # Preparing out-of-fold base-model predictions
    meta_train = np.zeros(
        (len(train_idx), n_models * n_classes),
        dtype=np.float32
    )

    meta_valid = np.zeros(
        (len(valid_idx), n_models * n_classes),
        dtype=np.float32
    )

    for model_idx, (name, estimator) in enumerate(
        stacking_base_estimators
    ):
        oof_predictions = np.zeros(
            (len(train_idx), n_classes),
            dtype=np.float32
        )

        # Generating group-aware out-of-fold predictions
        for inner_train_idx, inner_valid_idx in inner_cv.split(
            X_outer_train,
            y_outer_train,
            groups=groups_outer_train
        ):
            model = clone(estimator)

            model.fit(
                X_outer_train[inner_train_idx],
                y_outer_train[inner_train_idx]
            )

            predictions = model.predict(
                X_outer_train[inner_valid_idx]
            )

            oof_predictions[
                inner_valid_idx,
                predictions
            ] = 1.0

        start_col = model_idx * n_classes
        end_col = start_col + n_classes

        meta_train[:, start_col:end_col] = oof_predictions

        # Fitting base estimator on full outer-training fold
        final_base_model = clone(estimator)
        final_base_model.fit(X_outer_train, y_outer_train)

        valid_predictions = final_base_model.predict(
            X_outer_valid
        )

        meta_valid[
            np.arange(len(valid_idx)),
            start_col + valid_predictions
        ] = 1.0

    # Training the meta-classifier
    meta_model = clone(stacking_meta_estimator)
    meta_model.fit(meta_train, y_outer_train)

    # Predicting outer-validation fold
    y_pred = meta_model.predict(meta_valid)

    stacking_fold_results.append({
        "accuracy": accuracy_score(y_outer_valid, y_pred),
        "f1_macro": f1_score(
            y_outer_valid, y_pred, average="macro"
        ),
        "f1_weighted": f1_score(
            y_outer_valid, y_pred, average="weighted"
        )
    })

    print(f"Fold {fold}/5 completed.")

# Summarizing cross-validation performance
stacking_cv_df = pd.DataFrame(stacking_fold_results)

stacking_char_results = {
    "Model": "Stacking Classifier",
    "Feature_Representation": "Character TF-IDF",
    "Accuracy": stacking_cv_df["accuracy"].mean(),
    "Macro_F1": stacking_cv_df["f1_macro"].mean(),
    "Weighted_F1": stacking_cv_df["f1_weighted"].mean()
}

# Displaying individual model results
stacking_char_results_df = pd.DataFrame([stacking_char_results])

print("\nStacking Classifier (Character TF-IDF) Results:")
display(stacking_char_results_df.round(4))

# Updating centralized model comparison
model_results.append(stacking_char_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(by="Macro_F1", ascending=False)
    .reset_index(drop=True)
)

print("\nUpdated Model Comparison:")
display(model_results_df.round(4))

print("\nStacking Classifier cross-validation completed successfully.")

Fold 1/5 completed.
Fold 2/5 completed.
Fold 3/5 completed.
Fold 4/5 completed.
Fold 5/5 completed.

Stacking Classifier (Character TF-IDF) Results:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Stacking Classifier,Character TF-IDF,0.9872,0.9871,0.9872



Updated Model Comparison:


,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Ridge Classifier,Character TF-IDF,0.9878,0.9877,0.9878
1,Stacking Classifier,Character TF-IDF,0.9872,0.9871,0.9872
2,Ridge Classifier,Combined TF-IDF,0.9866,0.9864,0.9866
3,Voting Classifier,Character TF-IDF,0.9862,0.9860,0.9862
4,Linear SVM,Character TF-IDF,0.9860,0.9857,0.9860
5,MLP Classifier,Character TF-IDF,0.9847,0.9845,0.9847
6,MLP Classifier,Combined TF-IDF,0.9845,0.9843,0.9845
7,Linear SVM,Combined TF-IDF,0.9840,0.9837,0.9840
8,SGD Classifier,Character TF-IDF,0.9822,0.9822,0.9822
9,SGD Classifier,Combined TF-IDF,0.9823,0.9822,0.9823



Stacking Classifier cross-validation completed successfully.


**Observations on Stacking Classifier (Character TF-IDF):**

- Stacking Classifier achieved **98.72% accuracy** and a **Macro F1-score of 0.9871** using character-level TF-IDF features.
- The ensemble outperformed Voting Classifier (**0.9860 Macro F1**) and Linear SVM (**0.9857 Macro F1**), demonstrating the benefit of a learned meta-classifier over majority voting.
- Despite its competitive performance, Stacking Classifier remained slightly below Ridge Classifier (**0.9877 Macro F1**), indicating limited additional gains from ensemble complexity.

#### Logging Stacking Classifier (Character TF-IDF) in MLflow

In [69]:
# Logging Stacking Classifier with Character TF-IDF
with mlflow.start_run(run_name="StackingClassifier_Character_TFIDF"):

    # Logging ensemble configuration
    mlflow.log_params({
        "model": "StackingClassifier",
        "feature_representation": "Character TF-IDF",
        "base_estimators": "RidgeClassifier, LinearSVC, LogisticRegression",
        "meta_estimator": "LogisticRegression",
        "meta_features": "One-hot encoded class predictions",
        "ridge_alpha": 1.0,
        "linear_svm_max_iter": 5000,
        "logistic_regression_max_iter": 1000,
        "outer_cv_strategy": "StratifiedGroupKFold",
        "outer_cv_folds": 5,
        "inner_cv_strategy": "StratifiedGroupKFold",
        "inner_cv_folds": 3,
        "random_state": RANDOM_STATE
    })

    # Logging cross-validation metrics
    for metric in ["accuracy", "f1_macro", "f1_weighted"]:
        values = stacking_cv_df[metric]

        mlflow.log_metric(f"cv_{metric}_mean", values.mean())
        mlflow.log_metric(f"cv_{metric}_std", values.std())

    # Logging individual ensemble results
    mlflow.log_table(
        stacking_char_results_df,
        artifact_file="stacking_classifier_char_results.json"
    )

    # Logging fold-level results
    mlflow.log_table(
        stacking_cv_df,
        artifact_file="stacking_cv_fold_results.json"
    )

    # Logging updated model comparison
    mlflow.log_table(
        model_results_df,
        artifact_file="model_comparison.json"
    )

print("Stacking Classifier (Character TF-IDF) logged successfully in MLflow.")

Stacking Classifier (Character TF-IDF) logged successfully in MLflow.


### Hyperparameter Tuning

Our first tuning candidate is Ridge Classifier with Character TF-IDF, currently our strongest model with a cross-validation Macro F1 of 0.9877.

#### Configuring Ridge Classifier Hyperparameter Tuning

In [75]:
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# Defining the hyperparameter search space
ridge_param_grid = {
    "alpha": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}

# Configuring group-aware cross-validation
ridge_tuning_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Initializing Ridge Classifier
ridge_tuning_model = RidgeClassifier()

# Configuring grid search
ridge_grid_search = GridSearchCV(
    estimator=ridge_tuning_model,
    param_grid=ridge_param_grid,
    scoring="f1_macro",
    cv=ridge_tuning_cv,
    n_jobs=2,
    refit=True,
    return_train_score=False,
    verbose=1
)

print("Ridge Classifier Hyperparameter Tuning Configuration")
print(f"Parameter combinations: {len(ridge_param_grid['alpha'])}")
print(f"Cross-validation folds: {ridge_tuning_cv.n_splits}")
print(f"Total model fits: {len(ridge_param_grid['alpha']) * ridge_tuning_cv.n_splits}")
print("Scoring metric: Macro F1")
print("Feature representation: Character TF-IDF")

Ridge Classifier Hyperparameter Tuning Configuration
Parameter combinations: 6
Cross-validation folds: 5
Total model fits: 30
Scoring metric: Macro F1
Feature representation: Character TF-IDF


#### Tuning Ridge Classifier with Character TF-IDF

In [76]:
# Running group-aware hyperparameter tuning
ridge_grid_search.fit(
    X_train_char,
    y_train,
    groups=train_groups
)

# Extracting tuning results
ridge_tuning_results_df = pd.DataFrame(
    ridge_grid_search.cv_results_
)[
    [
        "param_alpha",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

# Renaming columns
ridge_tuning_results_df.columns = [
    "Alpha",
    "Mean_Macro_F1",
    "Std_Macro_F1",
    "Rank"
]

# Sorting by performance
ridge_tuning_results_df = (
    ridge_tuning_results_df
    .sort_values("Rank")
    .reset_index(drop=True)
)

# Displaying tuning results
print("Ridge Classifier Hyperparameter Tuning Results:")
display(ridge_tuning_results_df.round(4))

print(f"\nBest Alpha: {ridge_grid_search.best_params_['alpha']}")
print(f"Best CV Macro F1: {ridge_grid_search.best_score_:.4f}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Ridge Classifier Hyperparameter Tuning Results:


,Alpha,Mean_Macro_F1,Std_Macro_F1,Rank
0,0.5,0.9879,0.0021,1
1,1.0,0.9877,0.0021,2
2,0.1,0.9871,0.0026,3
3,2.0,0.9860,0.0024,4
4,5.0,0.9819,0.0033,5
5,10.0,0.9766,0.0031,6



Best Alpha: 0.5
Best CV Macro F1: 0.9879


**Observations on Ridge Classifier Hyperparameter Tuning:**

- The best regularization parameter was **alpha = 0.5**, achieving a cross-validation Macro F1-score of **0.9879 ± 0.0021**.
- Hyperparameter tuning produced a marginal improvement over the original Ridge Classifier (**0.9877**), indicating that the baseline configuration was already highly effective.
- Performance declined with stronger regularization, particularly at **alpha = 5.0 and 10.0**, suggesting that excessive regularization reduces predictive performance.

#### Logging Tuned Ridge Classifier in MLflow

In [77]:
# Logging tuned Ridge Classifier results
with mlflow.start_run(
    run_name="Ridge_Classifier_Char_TFIDF_Tuned"
):

    # Logging model configuration
    mlflow.log_params({
        "model": "RidgeClassifier",
        "feature_representation": "Character TF-IDF",
        "tuning_method": "GridSearchCV",
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5,
        "scoring": "f1_macro",
        "best_alpha": ridge_grid_search.best_params_["alpha"],
        "parameter_candidates": len(
            ridge_param_grid["alpha"]
        ),
        "random_state": 42
    })

    # Logging cross-validation metrics
    mlflow.log_metrics({
        "cv_macro_f1_mean": ridge_grid_search.best_score_,
        "cv_macro_f1_std": ridge_grid_search.cv_results_[
            "std_test_score"
        ][ridge_grid_search.best_index_]
    })

    # Logging hyperparameter tuning results
    mlflow.log_table(
        ridge_tuning_results_df,
        artifact_file="ridge_tuning_results.json"
    )

print("Tuned Ridge Classifier logged successfully in MLflow.")

Tuned Ridge Classifier logged successfully in MLflow.


#### Configuring Linear SVM Hyperparameter Tuning

In [78]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# Defining the hyperparameter search space
svm_param_grid = {
    "C": [0.01, 0.1, 0.5, 1.0, 2.0]
}

# Configuring group-aware cross-validation
svm_tuning_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Initializing Linear SVM
svm_tuning_model = LinearSVC(
    max_iter=5000,
    random_state=42
)

# Configuring grid search
svm_grid_search = GridSearchCV(
    estimator=svm_tuning_model,
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=svm_tuning_cv,
    n_jobs=2,
    refit=True,
    return_train_score=False,
    verbose=1
)

print("Linear SVM Hyperparameter Tuning Configuration")
print(f"Parameter combinations: {len(svm_param_grid['C'])}")
print(f"Cross-validation folds: {svm_tuning_cv.n_splits}")
print(f"Total model fits: {len(svm_param_grid['C']) * svm_tuning_cv.n_splits}")
print("Scoring metric: Macro F1")
print("Feature representation: Character TF-IDF")

Linear SVM Hyperparameter Tuning Configuration
Parameter combinations: 5
Cross-validation folds: 5
Total model fits: 25
Scoring metric: Macro F1
Feature representation: Character TF-IDF


#### Tuning Linear SVM with Character TF-IDF

In [80]:
# Running group-aware hyperparameter tuning
svm_grid_search.fit(
    X_train_char,
    y_train,
    groups=train_groups
)

# Extracting tuning results
svm_tuning_results_df = pd.DataFrame(
    svm_grid_search.cv_results_
)[
    [
        "param_C",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

# Renaming columns
svm_tuning_results_df.columns = [
    "C",
    "Mean_Macro_F1",
    "Std_Macro_F1",
    "Rank"
]

# Sorting by performance
svm_tuning_results_df = (
    svm_tuning_results_df
    .sort_values("Rank")
    .reset_index(drop=True)
)

# Displaying tuning results
print("Linear SVM Hyperparameter Tuning Results:")
display(svm_tuning_results_df.round(4))

print(f"\nBest C: {svm_grid_search.best_params_['C']}")
print(f"Best CV Macro F1: {svm_grid_search.best_score_:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Linear SVM Hyperparameter Tuning Results:


,C,Mean_Macro_F1,Std_Macro_F1,Rank
0,1.00,0.9857,0.0029,1
1,2.00,0.9853,0.0027,2
2,0.50,0.9850,0.0028,3
3,0.10,0.9794,0.0030,4
4,0.01,0.9450,0.0048,5



Best C: 1.0
Best CV Macro F1: 0.9857


**Observations on Linear SVM Hyperparameter Tuning:**

- The optimal regularization parameter was **C = 1.0**, achieving a cross-validation Macro F1-score of **0.9857 ± 0.0029**.
- Hyperparameter tuning did not improve upon the original Linear SVM performance, confirming that the default regularization strength was already effective.
- Stronger regularization significantly reduced performance, particularly at **C = 0.01**, where Macro F1 declined to **0.9450**.

#### Logging Tuned Linear SVM in MLflow

In [81]:
# Logging tuned Linear SVM results
with mlflow.start_run(
    run_name="Linear_SVM_Char_TFIDF_Tuned"
):

    # Logging model configuration
    mlflow.log_params({
        "model": "LinearSVC",
        "feature_representation": "Character TF-IDF",
        "tuning_method": "GridSearchCV",
        "cv_strategy": "StratifiedGroupKFold",
        "cv_folds": 5,
        "scoring": "f1_macro",
        "best_C": svm_grid_search.best_params_["C"],
        "parameter_candidates": len(svm_param_grid["C"]),
        "max_iter": 5000,
        "random_state": 42
    })

    # Logging cross-validation metrics
    mlflow.log_metrics({
        "cv_macro_f1_mean": svm_grid_search.best_score_,
        "cv_macro_f1_std": svm_grid_search.cv_results_[
            "std_test_score"
        ][svm_grid_search.best_index_]
    })

    # Logging hyperparameter tuning results
    mlflow.log_table(
        svm_tuning_results_df,
        artifact_file="svm_tuning_results.json"
    )

print("Tuned Linear SVM logged successfully in MLflow.")

Tuned Linear SVM logged successfully in MLflow.


#### Comparing Leading Models After Hyperparameter Tuning

In [82]:
# Preparing leading model comparison
leading_models_comparison_df = pd.DataFrame([
    {
        "Model": "Ridge Classifier (Tuned)",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": ridge_grid_search.best_score_
    },
    {
        "Model": "Ridge Classifier",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": 0.9877
    },
    {
        "Model": "Stacking Classifier",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": 0.9871
    },
    {
        "Model": "Linear SVM (Tuned)",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": svm_grid_search.best_score_
    },
    {
        "Model": "Linear SVM",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": 0.9857
    },
    {
        "Model": "Voting Classifier",
        "Feature_Representation": "Character TF-IDF",
        "CV_Macro_F1": 0.9860
    }
])

# Ranking models by cross-validation performance
leading_models_comparison_df = (
    leading_models_comparison_df
    .sort_values(
        "CV_Macro_F1",
        ascending=False
    )
    .reset_index(drop=True)
)

leading_models_comparison_df.index += 1

print("Leading Model Comparison:")
display(
    leading_models_comparison_df.round(4)
)

# Identifying the strongest model
best_model_name = leading_models_comparison_df.iloc[0]["Model"]
best_model_score = leading_models_comparison_df.iloc[0]["CV_Macro_F1"]

print(f"\nBest Model: {best_model_name}")
print(f"Best CV Macro F1: {best_model_score:.4f}")

Leading Model Comparison:


,Model,Feature_Representation,CV_Macro_F1
1,Ridge Classifier (Tuned),Character TF-IDF,0.9879
2,Ridge Classifier,Character TF-IDF,0.9877
3,Stacking Classifier,Character TF-IDF,0.9871
4,Voting Classifier,Character TF-IDF,0.9860
5,Linear SVM,Character TF-IDF,0.9857
6,Linear SVM (Tuned),Character TF-IDF,0.9857



Best Model: Ridge Classifier (Tuned)
Best CV Macro F1: 0.9879


**Observations on Final Model Comparison:**

- The **tuned Ridge Classifier with Character TF-IDF** achieved the highest cross-validation Macro F1-score of **0.9879**, outperforming all other evaluated configurations.
- The untuned Ridge Classifier (**0.9877**) and Stacking Classifier (**0.9871**) performed comparably, indicating that simpler linear models are highly competitive for product-title classification.
- The tuned Ridge Classifier was selected for final evaluation due to its leading predictive performance, computational efficiency, and relatively simple architecture.